# Loan Restructuring & Master Data Consolidation Pipeline

## Data Loading

In [ ]:
import pandas as pd
import numpy as np
import re
import os
from rapidfuzz import process, utils

# File Paths
file_3_5 = 'Table No 3.5 Population Group and Bank Group-wise Classification of Outstanding Credit of SCBs According to Occupation.xlsx'
file_restructuring = '13.Loan Subjected to Restructuring and Corporate Debt Restructured.xlsx'
file_npa = '_6.Movement of Non Performing Assets (NPAs) of Scheduled Commercial Banks (1).xlsx'

# Load Raw Data
table_3_5 = pd.read_excel(file_3_5)
table_restructuring = pd.read_excel(file_restructuring)
table_npa = pd.read_excel(file_npa)

print("Data loaded successfully.")

## Initial Inspection

In [ ]:
def inspect_table(df, name):
    print(f"--- Inspection: {name} ---")
    print(f"Shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()[:5]}...")
    display(df.head(3))

inspect_table(table_3_5, "Table 3.5")
inspect_table(table_restructuring, "Restructuring")
inspect_table(table_npa, "NPA Movement")

## Cleaning Utilities

In [ ]:
def to_camel_case(text):
    if pd.isna(text) or text == "": return "unnamedColumn"
    text = str(text)
    words = re.findall(r'[A-Z]?[a-z0-9]+|[A-Z]+(?=[A-Z][a-z0-9]|\b)', text)
    if not words: words = re.sub(r'[^a-zA-Z0-9]', ' ', text).split()
    if not words: return "unnamedColumn"
    processed = [words[0].lower()]
    for word in words[1:]:
        processed.append(word.capitalize())
    return "".join(processed)

def cleanse_bank_name(val):
    if pd.isna(val): return ""
    s = str(val).upper()
    s = re.sub(r'[^A-Z0-9 ]', '', s)
    return s.strip()

def extract_fiscal_year(val):
    if pd.isna(val): return None
    s = str(val).strip()
    match = re.findall(r'20(\d{2})', s)
    if match:
        return int("20" + match[-1])
    return None


## First Column Validation & Fix

In [ ]:
def validate_first_column(df):
    if df.empty: return df
    first_col = df.columns[0]
    if df[first_col].isna().all():
        df = df.drop(columns=[first_col])
        print("Dropped blank first column.")
    elif pd.api.types.is_numeric_dtype(df[first_col]):
        mean_val = df[first_col].mean()
        df[first_col] = df[first_col].fillna(mean_val)
        print(f"Filled numeric NaNs in first column with mean: {mean_val}")
    return df

table_3_5 = validate_first_column(table_3_5)
table_restructuring = validate_first_column(table_restructuring)
table_npa = validate_first_column(table_npa)

## Unnamed Column Renaming

In [ ]:
def infer_column_names(df):
    new_columns = list(df.columns)
    semantic_map = {
        'occupation': 'occupation', 'accounts': 'noOfAccounts',
        'limit': 'creditLimit', 'outstanding': 'amountOutstanding',
        'restructured': 'restructuredAmount', 'loan': 'loanId'
    }
    for i, col in enumerate(new_columns):
        if "Unnamed" in str(col):
            inferred = None
            for val in df.iloc[:15, i]:
                val_str = str(val).lower()
                for key, mapped in semantic_map.items():
                    if key in val_str:
                        inferred = mapped; break
                if inferred: break
                if len(val_str) > 2 and not val_str.replace('.','').isdigit():
                    inferred = val_str.strip(); break
            if inferred: new_columns[i] = inferred
    df.columns = new_columns
    return df

table_3_5 = infer_column_names(table_3_5)
table_restructuring = infer_column_names(table_restructuring)
table_npa = infer_column_names(table_npa)

## Row-Level Cleaning

In [ ]:
def clean_row_levels(df):
    if 1 in df.index:
        row_1_vals = df.loc[1]
        new_cols = list(df.columns)
        for i, val in enumerate(row_1_vals):
            if pd.notna(val) and str(val).strip() != "" and ("Unnamed" in str(new_cols[i]) or "unnamed" in str(new_cols[i]).lower()):
                new_cols[i] = str(val).strip()
        df.columns = new_cols
    if 2 in df.index:
        df = df.drop(index=2)
    return df

table_3_5 = clean_row_levels(table_3_5)
table_restructuring = clean_row_levels(table_restructuring)
table_npa = clean_row_levels(table_npa)

## Column Name Standardization

In [ ]:
def standardize_columns(df):
    df.columns = [to_camel_case(col) for col in df.columns]
    new_cols = []
    counts = {}
    for col in df.columns:
        if col in counts:
            counts[col] += 1
            new_cols.append(f"{col}_{counts[col]}")
        else:
            counts[col] = 0
            new_cols.append(col)
    df.columns = new_cols
    return df

table_3_5 = standardize_columns(table_3_5)
table_restructuring = standardize_columns(table_restructuring)
table_npa = standardize_columns(table_npa)

## Final Cleaned Output

In [ ]:
# Sub-Process 1.1: Temporal Normalization
def apply_temporal(df, name):
    year_col = next((col for col in df.columns if any(x in col.lower() for x in ['year', 'march', 'unnamed'])), df.columns[0])
    df['fiscalYear'] = df[year_col].apply(extract_fiscal_year).ffill()
    df = df[df['fiscalYear'] >= 2018].copy()
    df['fiscalYear'] = df['fiscalYear'].astype(int)
    print(f"Unique fiscalYear values for {name}: {sorted(df['fiscalYear'].unique())}")
    return df

table_3_5 = apply_temporal(table_3_5, "Table 3.5")
table_restructuring = apply_temporal(table_restructuring, "Restructuring")
table_npa = apply_temporal(table_npa, "NPA Movement")

# Sub-Process 1.2: Entity Resolution
bank_col_npa = next(col for col in table_npa.columns if any(x in col.lower() for x in ['bank', 'scheduled']))
table_npa['bankName'] = table_npa[bank_col_npa].map(cleanse_bank_name)
master_bank_list = [b for b in table_npa['bankName'].unique() if len(b) > 3]

def resolve_banks(df, master_list):
    df = df.copy()
    bank_col = next((col for col in df.columns if 'bank' in col.lower() and col != 'bankName'), df.columns[1])
    df['rawBankName'] = df[bank_col].map(cleanse_bank_name)
    unique_names = [n for n in df['rawBankName'].unique() if n]
    mapping = {name: process.extractOne(name, master_list, processor=utils.default_process)[0] 
               if name and len(name) > 3 and process.extractOne(name, master_list, processor=utils.default_process)[1] > 80 
               else name for name in unique_names if name}
    df['bankName'] = df['rawBankName'].map(mapping)
    return df

table_restructuring = resolve_banks(table_restructuring, master_bank_list)

# Sub-Process 1.3: Sectoral Aggregation
def aggregate_3_5(df):
    val_cols = [col for col in df.columns if any(x in col.lower() for x in ['outstanding', 'limit'])]
    id_cols = ['fiscalYear', 'occupation']
    melted = pd.melt(df, id_vars=id_cols, value_vars=val_cols, var_name='attr', value_name='val')
    melted['bankGroup'] = melted['attr'].apply(lambda x: 'Public' if 'public' in x.lower() else ('Private' if 'private' in x.lower() else 'Foreign'))
    features = melted.pivot_table(index=['fiscalYear', 'bankGroup'], columns='occupation', values='val', aggfunc='sum').reset_index()
    features.columns = [to_camel_case(f"credit_{c}") if c not in ['fiscalYear', 'bankGroup'] else c for c in features.columns]
    return features

df_3_5_features = aggregate_3_5(table_3_5)

# Sub-Process 1.4: Incremental Left-Join
def assign_group(name):
    if any(x in str(name).upper() for x in ['STATE BANK', 'CANARA', 'PUNJAB', 'INDIAN', 'BARODA', 'CENTRAL']): return 'Public'
    return 'Private'

table_npa['bankGroup'] = table_npa['bankName'].apply(assign_group)

rest_col = next((col for col in table_restructuring.columns if 'restructured' in col.lower()), None)
table_rest_subset = table_restructuring[['fiscalYear', 'bankName', rest_col]] if rest_col else table_restructuring[['fiscalYear', 'bankName']]

master_df = pd.merge(table_npa, table_rest_subset, on=['fiscalYear', 'bankName'], how='left')
master_df = pd.merge(master_df, df_3_5_features, on=['fiscalYear', 'bankGroup'], how='left')
master_df = master_df.loc[:, ~master_df.columns.duplicated()]

# Sub-Process 1.5: Missing Value Propagation
sector_cols = [c for c in master_df.columns if c.startswith('credit')]
master_df['isImputed'] = 0
if rest_col in master_df.columns:
    master_df.loc[master_df[rest_col].isnull(), 'isImputed'] = 1
for col in sector_cols:
    mask = master_df[col].isnull()
    master_df.loc[mask, 'isImputed'] = 1
    master_df[col] = master_df[col].fillna(master_df.groupby(['fiscalYear', 'bankGroup'])[col].transform('mean'))

# Sub-Process 1.6: Final Schema Validation
num_cols = master_df.select_dtypes(include=[np.number]).columns
master_df[num_cols] = master_df[num_cols].astype(np.float32)

# MANDATORY ASSERTIONS
assert master_df['bankName'].notnull().all(), "bankName contains nulls!"
assert master_df['fiscalYear'].notnull().all(), "fiscalYear contains nulls!"

master_df.to_csv('Master_Bank_Data_Consolidated.csv', index=False)
print("Master_Bank_Data_Consolidated.csv saved successfully.")
display(master_df.head())